# Microsoft Agent Framework による Agent 開発 (C#)

この notebook は [3.1_agentframework_csharp.md](./3.1_agentframework_csharp.md) の内容を、実行しながら確認できる形にまとめたものです。

- Microsoft Foundry の Project endpoint を使います
- 各セクションは個別に実行できます

In [ ]:
#r "nuget: Microsoft.Agents.AI.Foundry, 1.1.0"
#r "nuget: Microsoft.Agents.AI.Workflows, 1.1.0"
#r "nuget: Microsoft.Extensions.AI, 10.5.0"
#r "nuget: Azure.Identity, 1.20.0"
#r "nuget: DotNetEnv, 3.1.1"

In [ ]:
using System.ComponentModel;
using Azure.AI.Projects;
using Azure.Identity;
using DotNetEnv;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;
using Microsoft.Extensions.AI;

try
{
    Env.Load();  // .env を使う場合は最初に実行
    Console.WriteLine(".env loaded");
}
catch
{
    Console.WriteLine(".env file not found; environment variables from shell will be used.");
}

var foundryProjectEndpoint = Environment.GetEnvironmentVariable("FOUNDRY_PROJECT_ENDPOINT");
var foundryModel = Environment.GetEnvironmentVariable("FOUNDRY_MODEL") ?? "gpt-4.1";

Console.WriteLine($"FOUNDRY_PROJECT_ENDPOINT: {(string.IsNullOrWhiteSpace(foundryProjectEndpoint) ? "NOT SET" : "SET")}");
Console.WriteLine($"FOUNDRY_MODEL: {foundryModel}");

## 共通セットアップ

以下のセルで Foundry に接続するための共通関数を用意します。

In [ ]:
AIProjectClient GetProjectClient()
{
    if (string.IsNullOrWhiteSpace(foundryProjectEndpoint))
    {
        throw new InvalidOperationException("FOUNDRY_PROJECT_ENDPOINT が未設定です。.env を使う場合は DotNetEnv.Env.Load() を実行してください。");
    }

    return new AIProjectClient(new Uri(foundryProjectEndpoint), new DefaultAzureCredential());
}

## 1. シンプルなエージェント

In [ ]:
AIAgent helloAgent = GetProjectClient().AsAIAgent(
    model: foundryModel,
    instructions: "あなたは親切な AI アシスタントです。日本語で簡潔に回答してください。",
    name: "HelloAgent");

Console.WriteLine(await helloAgent.RunAsync("Microsoft Foundry とは何ですか？ひとことで教えてください。"));

## 2. 関数ツールの追加

In [ ]:
[Description("Get the weather for a given location.")]
static string GetWeather([Description("The location to get the weather for.")] string location)
    => $"{location} の天気は晴れ、最高気温 25°C です。";

AITool weatherTool = AIFunctionFactory.Create(GetWeather);

AIAgent weatherAgent = GetProjectClient().AsAIAgent(
    model: foundryModel,
    instructions: "あなたは天気案内アシスタントです。天気を聞かれたら GetWeather ツールを使って答えてください。",
    name: "WeatherAssistant",
    tools: [weatherTool]);

AgentSession weatherSession = await weatherAgent.CreateSessionAsync();
Console.WriteLine(await weatherAgent.RunAsync("東京の天気を教えてください。", weatherSession));

## 3. 複数ターンの会話

In [ ]:
AIAgent conversationAgent = GetProjectClient().AsAIAgent(
    model: foundryModel,
    instructions: "あなたは会話の文脈を理解する AI アシスタントです。日本語で簡潔に回答してください。",
    name: "ConversationAgent");

AgentSession session = await conversationAgent.CreateSessionAsync();

Console.WriteLine(await conversationAgent.RunAsync("私の名前は Ayako で、趣味は登山です。", session));
Console.WriteLine();
Console.WriteLine(await conversationAgent.RunAsync("私について覚えていることを教えてください。", session));

## 4. マルチエージェントワークフロー

In [ ]:
AIAgent writer = GetProjectClient().AsAIAgent(
    model: foundryModel,
    instructions: "ユーザーの依頼に対して、簡潔な説明文を日本語で作成してください。",
    name: "writer");

AIAgent reviewer = GetProjectClient().AsAIAgent(
    model: foundryModel,
    instructions: "直前の回答をレビューし、より分かりやすい改善版を日本語で返してください。",
    name: "reviewer");

AIAgent workflowAgent = AgentWorkflowBuilder.BuildSequential(writer, reviewer).AsAIAgent();

string? lastAuthor = null;
await foreach (var update in workflowAgent.RunStreamingAsync("Microsoft Foundry の特徴を 3 つ挙げてください。"))
{
    if (string.IsNullOrEmpty(update.Text))
    {
        continue;
    }

    if (lastAuthor != update.AuthorName)
    {
        lastAuthor = update.AuthorName;
        Console.WriteLine();
        Console.WriteLine($"[{update.AuthorName}]");
    }

    Console.Write(update.Text);
}
Console.WriteLine();